# 09. Hybrid RRF Retrieval & Generation Evaluation Benchmark

This notebook demonstrates the quantitative evaluation framework used to score the Hybrid Reciprocal Rank Fusion (RRF) retrieval strategy across retrieval metrics (MRR, nDCG, Keyword Coverage) and LLM-as-a-Judge answer quality (`src/step8_evals_hybrid.py`).

### Key Demonstration Steps:
1. **Global BM25 + Vector Retriever Caching**: Initializing `_ALL_CHUNKS` and `_BM25_RETRIEVER` once to prevent redundant file I/O overhead across evaluation runs.
2. **Hybrid RRF Evaluation Flow**: Executing `get_hybrid_chunks` (combining BM25 lexical and Dense vector searches) alongside `answer_question` with `retriever_type="hybrid"`.
3. **Metric Comparison Framework**: Benchmarking Hybrid RRF performance outputs for comparison against the baseline Dense retriever results.

In [1]:
import sys
import os
import pandas as pd

# Add project root to path for modular imports
sys.path.append("..")

from src.step8_evals_hybrid import (
    load_test_dataset, 
    run_single_eval, 
    get_hybrid_chunks,
    TestItem,
    PROCESSED_DATA_DIR,
    RESULTS_DIR
)

## Step 1: Benchmark Dataset Initialization

Load evaluation test items from `tests.jsonl` to execute hybrid evaluation runs.

In [2]:
test_file = PROCESSED_DATA_DIR / "tests.jsonl"
tests = load_test_dataset(test_file)

print(f"Loaded {len(tests)} evaluation items for Hybrid RRF benchmark execution.")

Loaded 274 evaluation tests from 'tests.jsonl'.
Loaded 274 evaluation items for Hybrid RRF benchmark execution.


## Step 2: Run Hybrid RRF Single-Item Evaluation

Execute `run_single_eval` on a sample test query to verify that BM25 and Dense retrievers are fused properly via Reciprocal Rank Fusion prior to scoring.

In [3]:
sample_test = tests[0]

print(f"Executing Hybrid RRF Evaluation for Test Item:")
print(f"• Question    : {sample_test.question}")
print(f"• Target File : {sample_test.source_file}\n")

# Verify hybrid retriever chunk extraction
retrieved_chunks = get_hybrid_chunks(sample_test.question, k=4)
print(f"Extracted {len(retrieved_chunks)} Hybrid RRF Chunks successfully.\n")

# Execute full single item evaluation pipeline
eval_result = run_single_eval(sample_test)

print("--- Evaluation Output Payload (Hybrid RRF) ---")
print(f"Source Found     : {eval_result['source_found']}")
print(f"MRR / nDCG       : {eval_result['mrr']:.2f} / {eval_result['ndcg']:.2f}")
print(f"Keyword Coverage : {eval_result['keyword_coverage']*100:.1f}% ({eval_result['keywords_found']})")
print(f"Accuracy Score   : {eval_result['accuracy']} / 5.0")
print(f"Completeness     : {eval_result['completeness']} / 5.0")
print(f"Relevance        : {eval_result['relevance']} / 5.0")
print(f"Judge Feedback   : {eval_result['judge_feedback']}")

Executing Hybrid RRF Evaluation for Test Item:
• Question    : What does the letter dictated by Shriji Maharaj emphasize about attaining a human birth in Bharat-khand?
• Target File : Gadhada_I_1.md

Extracted 4 Hybrid RRF Chunks successfully.

--- Evaluation Output Payload (Hybrid RRF) ---
Source Found     : False
MRR / nDCG       : 0.00 / 0.00
Keyword Coverage : 100.0% (5/5)
Accuracy Score   : 5.0 / 5.0
Completeness     : 5.0 / 5.0
Relevance        : 5.0 / 5.0
Judge Feedback   : The Generated Answer accurately captures the key points of the Reference Answer, emphasizing the rarity and significance of human birth in Bharat-khand, the comparison to chintamani, and the longing of deities for this birth. It also includes additional insights about spiritual growth and the importance of focusing on liberation, which enhances the completeness of the response.


## Step 3: Result Persistence & Path Verification

Verify configured output paths for saving hybrid evaluation CSV and JSON files in `RESULTS_DIR`.

In [4]:
csv_output = RESULTS_DIR / "eval_results_hybrid.csv"
json_output = RESULTS_DIR / "eval_results_hybrid.json"

print(f"Target Hybrid CSV Output Path  : {csv_output}")
print(f"Target Hybrid JSON Output Path : {json_output}")

Target Hybrid CSV Output Path  : C:\Users\Lenovo\projects\Active Vachanamrut RAG project\results\eval_results_hybrid.csv
Target Hybrid JSON Output Path : C:\Users\Lenovo\projects\Active Vachanamrut RAG project\results\eval_results_hybrid.json
